# 06 - My final regression and XGBoost analysis

I wait for all 1,000 audited answers from each model. My pooled regression retains the fixed board-characteristic specification and puzzle-clustered standard errors. My final predictive models are **XGBoost classifiers**, separately fitted on each deployment's 800 training puzzles and evaluated once on the same 200 held-out puzzle IDs.

I fixed 150 depth-3 boosted trees with learning rate 0.05 and regularization before any held-out evaluation; I perform no hyperparameter search or early stopping. `experiments/one_move_v1/final_analysis_plan.json` records this user-authorized amendment after the pilot. The earlier pilot CART figures are unchanged and remain descriptive.

In [1]:
MODE = 'final'
BOOK = '06_final_analysis.ipynb'

## 1. I check audited completion and freeze inputs

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tools.sm_exceptions import PerfectSeparationWarning
import xgboost
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, brier_score_loss, confusion_matrix
import joblib, nbformat
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'experiments/one_move_v1/config.json').exists())
BASE=ROOT/'data/experiments/one_move_v1';CFG=ROOT/'experiments/one_move_v1'
CONFIG=json.loads((CFG/'config.json').read_text());PLAN=json.loads((CFG/'final_analysis_plan.json').read_text())
assert xgboost.__version__==PLAN['xgboost_version'], 'Use the pinned XGBoost version.'
PREP=json.loads((BASE/'preparation.json').read_text());RECIPE=json.loads((BASE/'feature_recipe.json').read_text())
OUT=ROOT/'data/analysis/one_move_v1'/MODE;OUT.mkdir(parents=True,exist_ok=True)
FIG=ROOT/'figures/one_move_v1'/MODE

def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def save(p,value):Path(p).write_text(json.dumps(value,indent=2)+'\n',encoding='utf-8')
def code_sha(name):
    n=nbformat.read(ROOT/'notebooks'/name,as_version=4)
    return hashlib.sha256('\n\n'.join(c.source for c in n.cells if c.cell_type=='code' and 'run_control' not in c.metadata.get('tags',[])).encode()).hexdigest()
manifest=json.loads((BASE/'manifest.json').read_text())
for key,f in [('config_sha256','config.json'),('plan_sha256','analysis_plan.json'),('final_plan_sha256','final_analysis_plan.json'),('prompt_sha256','prompt.txt')]:assert sha(CFG/f)==manifest[key]
for key,f in [('collection_code_sha256','04_collect_answers.ipynb'),('analysis_code_sha256','06_final_analysis.ipynb'),('preparation_code_sha256','03_prepare_study.ipynb'),('early_code_sha256','05_early_results.ipynb')]:assert code_sha(f)==manifest[key]
assert sha(BASE/'puzzles.csv')==PREP['sample_sha256'] and sha(BASE/'analysis_features.csv')==PREP['feature_sha256']
assert sha(BASE/'feature_recipe.json')==PREP['recipe_sha256']
features=pd.read_csv(BASE/'analysis_features.csv',dtype={'puzzle_id':str})
assert len(features)==1000 and features.total_solver_moves.eq(1).all()
TARGET=100 if MODE=='early' else 1000
frames={};progress=[];metadata={}
for name,spec in CONFIG['models'].items():
    folder=BASE/name;path=folder/'results.csv';data=None;complete=0;target_complete=0
    if path.exists():
        audit=json.loads((folder/'audit.json').read_text())
        assert audit['passed'] and audit['sample_sha256']==PREP['sample_sha256'] and audit['results_sha256']==sha(path)
        assert audit['manifest_sha256']==sha(BASE/'manifest.json')
        assert audit['scoring_version']=='solution_mention_v1'
        assert audit['scoring_amendment_sha256']==sha(CFG/'scoring_amendment.json')
        result=pd.read_csv(path,dtype={'puzzle_id':str})
        assert len(result)==result.puzzle_id.nunique()==1000 and set(result.puzzle_id)==set(features.puzzle_id)
        selected=features.loc[features.pilot] if MODE=='early' else features
        cols=['puzzle_id','status','success','failure_type','answer','returned_model','trial_id']
        data=selected.merge(result[cols],on='puzzle_id',validate='one_to_one')
        complete=int(result.status.eq('completed').sum());target_complete=int(data.status.eq('completed').sum())
        if target_complete==TARGET:
            assert data.success.isin([0,1]).all()
            if MODE=='early':assert data.split.eq('train').all()
            frames[name]=data
            metadata[name]={'source_results_sha256':sha(path),'audit_sha256':sha(folder/'audit.json'),'n':len(data)}
    progress.append({'Model':spec['label'],'Completed overall':complete,'Ready cases':target_complete,'Required for this notebook':TARGET})
display(pd.DataFrame(progress))
READY=len(frames)==2
if not READY:
    save(OUT/'status.json',{'status':'waiting_for_both_pilots' if MODE=='early' else 'waiting_for_complete_collection','models':progress,'new_api_requests':0,'fits_created':False,'heldout_evaluation_performed':False})
    print('Final analysis is pending: I need all 1,000 audited answers from each model. I do not fit XGBoost or evaluate the holdout yet. No API calls.')
if READY:
    snap_path=OUT/'snapshot.json'
    if not snap_path.exists():
        for name,d in frames.items():
            d.to_csv(OUT/(name+'_inputs.csv'),index=False)
            metadata[name]['inputs_sha256']=sha(OUT/(name+'_inputs.csv'))
        snapshot={'mode':MODE,'created_utc':datetime.now(timezone.utc).isoformat(),'sample_sha256':PREP['sample_sha256'],
                  'feature_sha256':PREP['feature_sha256'],'plan_sha256':sha(CFG/'final_analysis_plan.json'),'code_sha256':code_sha(BOOK),'models':metadata}
        save(snap_path,snapshot)
    else:snapshot=json.loads(snap_path.read_text())
    assert snapshot['plan_sha256']==sha(CFG/'final_analysis_plan.json') and snapshot['code_sha256']==code_sha(BOOK)
    assert snapshot['sample_sha256']==PREP['sample_sha256'] and snapshot['feature_sha256']==PREP['feature_sha256']
    for name in CONFIG['models']:
        p=OUT/(name+'_inputs.csv');assert sha(p)==snapshot['models'][name]['inputs_sha256']
        frames[name]=pd.read_csv(p,dtype={'puzzle_id':str})
        if MODE=='early':assert frames[name].split.eq('train').all()
    print('I use frozen inputs. Later collection cannot overwrite this snapshot.')

,Model,Completed overall,Ready cases,Required for this notebook
0,GPT-OSS 120B,100,100,1000
1,Gemini 3.5 Flash-Lite,100,100,1000


Final analysis is pending: I need all 1,000 audited answers from each model. I do not fit XGBoost or evaluate the holdout yet. No API calls.


## 2. I fit the fixed regression and freeze XGBoost before test evaluation

In [3]:
def wilson(correct,n):
    p=correct/n;z=1.96;den=1+z*z/n;center=(p+z*z/(2*n))/den
    half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return [float(center-half),float(center+half)]
fit=None;boosters={};coefs=None
if READY:
    predictors=RECIPE['core'] if MODE=='early' else RECIPE['full']
    tree_predictors=[{'rating_500':'puzzle_rating','legal_moves_10':'legal_moves','piece_count_10':'piece_count'}.get(k,k) for k in predictors]
    pooled=pd.concat([d.assign(is_gemini=int(name=='gemini'),model_key=name) for name,d in frames.items()],ignore_index=True)
    names=predictors+['is_gemini'];X=sm.add_constant(pooled[names].astype(float),has_constant='add');y=pooled.success.astype(int)
    threshold=15 if MODE=='early' else 50
    supported=all(min(int(d.success.sum()),len(d)-int(d.success.sum()))>=threshold for d in frames.values())
    reasons=[]
    if not supported:reasons.append('Insufficient successes or failures in a model; I do not simplify the fixed model to force a fit.')
    elif np.linalg.matrix_rank(X.to_numpy())!=X.shape[1]:reasons.append('Fixed design matrix is rank deficient; no regression estimated.')
    else:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('error',PerfectSeparationWarning)
                fit=sm.GLM(y,X,family=sm.families.Binomial()).fit(cov_type='cluster',cov_kwds={'groups':pd.factorize(pooled.puzzle_id)[0]})
            assert fit.converged and np.isfinite(fit.params).all() and np.isfinite(fit.cov_params().to_numpy()).all()
            ci=fit.conf_int()
            coefs=pd.DataFrame({'term':fit.params.index,'coefficient':fit.params.values,'odds_ratio':np.exp(fit.params.values),
                               'or_ci_low':np.exp(ci[0].values),'or_ci_high':np.exp(ci[1].values),'p_value':fit.pvalues.values})
            assert np.isfinite(coefs.select_dtypes('number').to_numpy()).all()
            coefs.to_csv(OUT/'logistic_coefficients.csv',index=False);joblib.dump(fit,OUT/'logistic_regression.joblib')
            if MODE=='final':save(OUT/'primary_rating_test.json',{'two_sided_p':float(fit.pvalues['rating_500']),'cluster':'puzzle_id','scope':'Prespecified rating association; secondary coefficients are descriptive.'})
        except Exception as error:
            fit=None;reasons.append('Regression checks failed: '+type(error).__name__)

    rows=[];metrics=[]
    parameters=PLAN['xgboost_parameters']
    for name,d in frames.items():
        correct=int(d.success.sum())
        rows.append({'Model':CONFIG['models'][name]['label'],'Correct':correct,'Puzzles':len(d),'Accuracy':correct/len(d),'wilson_95':wilson(correct,len(d))})
        train=d.loc[d.split.eq('train')].copy();test=d.loc[d.split.eq('test')].copy()
        assert len(train)==800 and len(test)==200 and not set(train.puzzle_id)&set(test.puzzle_id)
        if min(int(train.success.sum()),len(train)-int(train.success.sum()))<30:
            reasons.append(name+': insufficient training class counts for XGBoost.');continue
        model_path=OUT/(name+'_xgboost.ubj');training_path=OUT/(name+'_xgboost_training.json')
        expected={'model_key':name,'method':'XGBoost','parameters':parameters,'features':tree_predictors,
                  'train_ids':sorted(train.puzzle_id),'source_input_sha256':sha(OUT/(name+'_inputs.csv')),
                  'final_plan_sha256':sha(CFG/'final_analysis_plan.json'),'xgboost_version':xgboost.__version__}
        if model_path.exists() or training_path.exists():
            assert model_path.exists() and training_path.exists(), 'Incomplete saved fit; inspect before evaluation.'
            training=json.loads(training_path.read_text())
            assert all(training[k]==v for k,v in expected.items()), 'Frozen training specification changed.'
            assert training['model_sha256']==sha(model_path)
            booster=XGBClassifier(**parameters);booster.load_model(model_path)
        else:
            booster=XGBClassifier(**parameters)
            # No eval_set, early stopping or parameter search: the held-out labels never enter fit.
            booster.fit(train[tree_predictors],train.success.astype(int))
            temporary=model_path.with_name(model_path.stem+'.tmp.ubj');booster.save_model(temporary);os.replace(temporary,model_path)
            training={**expected,'model_sha256':sha(model_path)}
            save(training_path,training)
        boosters[name]=booster
        gains=booster.get_booster().get_score(importance_type='gain');total=sum(gains.values())
        importance=pd.DataFrame({'feature':tree_predictors,'gain':[float(gains.get(k,0)) for k in tree_predictors]})
        importance['normalized_gain']=importance.gain/total if total else 0.
        importance_path=OUT/(name+'_xgboost_feature_importance.csv')
        if importance_path.exists():
            pd.testing.assert_frame_equal(pd.read_csv(importance_path),importance,check_dtype=False,rtol=1e-6)
        else:importance.to_csv(importance_path,index=False)
        # The model and training receipt are frozen before any held-out prediction.
        assert sha(model_path)==training['model_sha256']
        prediction_path=OUT/(name+'_heldout_predictions.csv');evaluation_path=OUT/(name+'_heldout_evaluation.json')
        if evaluation_path.exists():
            receipt=json.loads(evaluation_path.read_text())
            assert receipt['model_sha256']==sha(model_path) and receipt['input_sha256']==expected['source_input_sha256']
            assert receipt['predictions_sha256']==sha(prediction_path)
            cached=pd.read_csv(prediction_path,dtype={'puzzle_id':str});assert set(cached.puzzle_id)==set(test.puzzle_id) and len(cached)==200
            metric=receipt['metrics']
        else:
            assert not prediction_path.exists(), 'Partial evaluation exists; inspect before continuing.'
            probability=booster.predict_proba(test[tree_predictors])[:,1]
            prediction=(probability>=PLAN['class_threshold']).astype(int);truth=test.success.astype(int)
            ncorrect=int((prediction==truth).sum());majority=int(train.success.mean()>=.5)
            metric={'Model':CONFIG['models'][name]['label'],'method':'XGBoost','test_n':200,
                    'accuracy':float(accuracy_score(truth,prediction)),'wilson_95':wilson(ncorrect,200),
                    'balanced_accuracy':float(balanced_accuracy_score(truth,prediction)),
                    'roc_auc':float(roc_auc_score(truth,probability)) if truth.nunique()==2 else None,
                    'brier':float(brier_score_loss(truth,probability)),
                    'confusion_tn_fp_fn_tp':confusion_matrix(truth,prediction,labels=[0,1]).ravel().tolist(),
                    'majority_baseline_accuracy':float((truth==majority).mean())}
            pd.DataFrame({'puzzle_id':test.puzzle_id,'truth':truth,'probability':probability,'prediction':prediction}).to_csv(prediction_path,index=False)
            save(evaluation_path,{'model_sha256':sha(model_path),'input_sha256':expected['source_input_sha256'],
                                 'predictions_sha256':sha(prediction_path),'test_ids':sorted(test.puzzle_id),'metrics':metric})
        metrics.append(metric)
    pd.DataFrame(rows).to_csv(OUT/'accuracy.csv',index=False)
    if metrics:save(OUT/'heldout_metrics.json',metrics)
    report={'status':'analyzed','scope':'final','regression_fitted':fit is not None,'puzzle_clusters':1000,
            'prediction_method':'XGBoost','xgboost_models':list(boosters),'reasons':reasons,
            'heldout_evaluation_performed':bool(metrics),'new_api_requests':0,'accuracy':rows}
    save(OUT/'status.json',report);display(pd.DataFrame(rows))
    if coefs is not None:display(coefs)
    if metrics:display(pd.DataFrame(metrics)[['Model','accuracy','balanced_accuracy','roc_auc','brier','majority_baseline_accuracy']])
    for reason in reasons:print(reason)


## 3. My final accuracy, regression and XGBoost feature-importance figures

In [4]:
if READY:
    FIG.mkdir(parents=True,exist_ok=True)
    heading='Pilot: 100 training puzzles per model' if MODE=='early' else 'Final: assisted one-move puzzle accuracy'
    fig,ax=plt.subplots(figsize=(8,4))
    for i,row in enumerate(rows):
        low,high=row['wilson_95'];p=row['Accuracy'];ax.bar(i,p,width=.55,color=['#2563eb','#16a34a'][i])
        ax.errorbar(i,p,yerr=[[p-low],[high-p]],fmt='none',color='black',capsize=4)
        ax.text(i,min(.96,p+.10),f"{row['Correct']}/{row['Puzzles']} ({p:.1%})",ha='center')
    ax.set(xticks=[0,1],xticklabels=[r['Model'] for r in rows],ylim=(0,1),ylabel='Official solution-move accuracy',title=heading)
    fig.tight_layout();fig.savefig(FIG/'accuracy.png',dpi=180);plt.show()
    fig,ax=plt.subplots(figsize=(8,4.5))
    if fit is not None:
        ratings=np.linspace(pooled.puzzle_rating.min(),pooled.puzzle_rating.max(),200)
        reference={k:0. for k in predictors}
        for k in RECIPE['core']:reference[k]=float(features.loc[features.split.eq('train'),k].median())
        for name,color in [('gpt_oss','#2563eb'),('gemini','#16a34a')]:
            grid=pd.DataFrame({k:np.full(200,v) for k,v in reference.items()});grid['rating_500']=(ratings-1500)/500;grid['is_gemini']=int(name=='gemini')
            grid=sm.add_constant(grid,has_constant='add')[X.columns]
            prediction=fit.get_prediction(grid).summary_frame()
            ax.plot(ratings,prediction['mean'],color=color,label=CONFIG['models'][name]['label'])
            ax.fill_between(ratings,prediction['mean_ci_lower'],prediction['mean_ci_upper'],color=color,alpha=.12)
        ax.legend();ax.set(ylim=(0,1),xlabel='Human puzzle rating',ylabel='Predicted solution-move success',title='Exploratory adjusted curves' if MODE=='early' else 'Adjusted success curves')
        ax.text(.5,-.23,'Numeric board features held at training medians; theme indicators zero and reference phase.\nOrdinary 95% bands use puzzle-clustered standard errors.',transform=ax.transAxes,ha='center',fontsize=8)
    else:ax.text(.5,.5,'Regression not supported; see status explanation.',transform=ax.transAxes,ha='center');ax.set_axis_off()
    fig.tight_layout();fig.savefig(FIG/'logistic_regression.png',dpi=180);plt.show()

    fig,axes=plt.subplots(1,2,figsize=(12,5),sharex=True)
    labels={'puzzle_rating':'Human rating','legal_moves':'Legal-move count','piece_count':'Piece count','material_balance':'Material balance'}
    for ax,name,color in zip(axes,CONFIG['models'],['#2563eb','#16a34a']):
        path=OUT/(name+'_xgboost_feature_importance.csv')
        if name not in boosters:
            ax.text(.5,.5,'Training class-count gate not met',transform=ax.transAxes,ha='center');ax.set_axis_off();continue
        table=pd.read_csv(path).sort_values('normalized_gain')
        names=[labels.get(k,k.replace('theme_','Theme: ').replace('phase_','Phase: ')) for k in table.feature]
        ax.barh(names,table.normalized_gain,color=color)
        ax.set(title=CONFIG['models'][name]['label'],xlabel='Normalized training gain importance')
    fig.suptitle('XGBoost: board characteristics used for prediction\n150 boosted trees per model, fitted on 800 training puzzles')
    fig.text(.5,.01,'Training importance is descriptive; it does not establish causal effects or held-out importance.',ha='center',fontsize=9)
    fig.tight_layout(rect=(0,.035,1,.94));fig.savefig(FIG/'xgboost_feature_importance.png',dpi=180);plt.show()
    print('I saved final accuracy, regression and XGBoost feature-importance figures. The pilot CARTs remain in notebook 05.')


I make no API calls here. Missing collection remains pending. I reuse frozen fitted models and cached held-out evaluations on rerun. XGBoost training-gain importance is descriptive, not causal. My original PDFs and early pilot figures remain unchanged.